# Example 7.11.3 / 2: Disjunctive Formulation

Companion notebook for **7.11.3 Scheduling Formulations** in the AMPL lab Documentation, section "Disjunctive Formulation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Disjunctive Formulation

With continuous start times, a binary `Before[i,j]` chooses the order of each pair of jobs, and a
big-M pair of rows keeps them from overlapping (Big-M Formulations (AMPL lab documentation, section 7.3)). The constant
`M[i,j]` is derived from the data: `finish[i] - start[j]` can be at most
`horizon - release[j]`, so that value relaxes the row when the other order is chosen:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set JOBS ordered;                   # jobs to schedule
param duration {JOBS} > 0;          # processing time
param release {JOBS} >= 0;          # earliest start
param weight {JOBS} >= 0;           # importance of finishing early
param horizon > 0;                  # latest completion time
param M {i in JOBS, j in JOBS} := horizon - release[j];  # largest finish[i] - start[j]

var start {j in JOBS} >= release[j], <= horizon - duration[j];  # start time of the job
var finish {j in JOBS} = start[j] + duration[j];                 # completion time of the job
var Before {i in JOBS, j in JOBS: ord(i) < ord(j)} binary;       # 1 if i runs before j

minimize WeightedCompletion: sum {j in JOBS} weight[j] * finish[j];
# Before[i,j] = 1: job i finishes before job j starts
subject to IBeforeJ {i in JOBS, j in JOBS: ord(i) < ord(j)}:
    finish[i] <= start[j] + M[i,j] * (1 - Before[i,j]);
# Before[i,j] = 0: job j finishes before job i starts
subject to JBeforeI {i in JOBS, j in JOBS: ord(i) < ord(j)}:
    finish[j] <= start[i] + M[j,i] * Before[i,j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["JOBS"] = ["A", "B", "C"]
ampl.param["duration"] = {"A": 3, "B": 2, "C": 4}
ampl.param["release"] = {"A": 0, "B": 1, "C": 0}
ampl.param["weight"] = {"A": 1, "B": 3, "C": 2}
ampl.param["horizon"] = 12

In [5]:
# run.py
ampl.solve(solver="highs")
print("starts:", ampl.var["start"].to_dict())

HiGHS 1.14.0: optimal solution; objective 33
6 simplex iterations
1 branching nodes
starts: {'A': 7, 'B': 1, 'C': 3}


### Pure AMPL: data.dat and model.run

The disjunctive model finds the same schedule and the same objective, 33, with three binaries
instead of one per job and start period. It avoids a time grid but relies on valid, tight
horizon bounds. Create one ordering binary per unordered pair that can actually share a
resource, as `ord(i) < ord(j)` does here; precedence arcs can fix some orders in advance and
remove their binaries. Implications (`Before[i,j] = 1 ==> finish[i] <= start[j]`) can replace
the explicit big-M rows (Implication and Indicator Constraints (AMPL lab documentation, section 7.5)).

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: JOBS: duration release weight :=
  A  3  0  1
  B  2  1  3
  C  4  0  2 ;
param horizon := 12;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display start, finish;

HiGHS 1.14.0: optimal solution; objective 33
6 simplex iterations
1 branching nodes
: start finish    :=
A    7     10
B    1      3
C    3      7
;

